In [1]:
import os
import urllib.request
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [2]:
url = "http://aire.cdmx.gob.mx/descargas/Opendata/anuales_horarios/contaminantes_2026.csv"
filename = "contaminantes_2026.csv"

if not os.path.exists(filename):
    print(f"Descargando {filename} desde el portal del SIMAT...")
    urllib.request.urlretrieve(url, filename)
    print("Descarga finalizada con éxito.")
else:
    print(f"El archivo '{filename}' ya existe en el entorno local.")

Descargando contaminantes_2026.csv desde el portal del SIMAT...
Descarga finalizada con éxito.


In [3]:
df = pd.read_csv(filename, skiprows=9)

In [4]:
# 3. Inciso a): Tabla con un renglón por combinación (estación, hora/fecha) y una columna por contaminante
pivoted = df.pivot(
    index=["id_station", "date"], columns="id_parameter", values="valor"
)

print("==================================================")
print("a) TABLA REORGANIZADA (Renglón: Estación-Hora | Columna: Contaminante)")
print("==================================================")
print(f"Dimensiones de la matriz: {pivoted.shape}")
pivoted.head()

a) TABLA REORGANIZADA (Renglón: Estación-Hora | Columna: Contaminante)
Dimensiones de la matriz: (172992, 9)


id_parameter                    CO  NO  NO2  NOX  O3  PM10  PM2.5  PMCO  SO2
id_station date                                                             
ACO        2026-01-01 00:00:00 NaN NaN  NaN  NaN NaN   NaN    NaN   NaN  NaN
           2026-01-01 01:00:00 NaN NaN  NaN  NaN NaN   NaN    NaN   NaN  NaN
           2026-01-01 02:00:00 NaN NaN  NaN  NaN NaN   NaN    NaN   NaN  NaN
           2026-01-01 03:00:00 NaN NaN  NaN  NaN NaN   NaN    NaN   NaN  NaN
           2026-01-01 04:00:00 NaN NaN  NaN  NaN NaN   NaN    NaN   NaN  NaN

In [5]:
# b) Cálculo de Covarianza y Correlación entre CO y NOX
cov_co_nox = pivoted["CO"].cov(pivoted["NOX"])
corr_co_nox = pivoted["CO"].corr(pivoted["NOX"])

print("==================================================")
print("b) COVARIANZA Y CORRELACIÓN ENTRE CO Y NOX")
print("==================================================")
print(f"Covarianza Cov(CO, NOX)  : {cov_co_nox:.4f}")
print(f"Correlación Corr(CO, NOX): {corr_co_nox:.4f}\n")

print(
    """¿Son distintas?
Sí, son métricas distintas:
1. La COVARIANZA mide la dirección de la relación lineal entre las variables en las unidades originales de medición (ppm × ppb). Su magnitud depende totalmente de las escalas de CO y NOX.
2. La CORRELACIÓN es una versión estandarizada/adimensional de la covarianza (restringida al intervalo [-1, 1]). Permite comparar la fuerza de la asociación lineal de manera independiente a las unidades de medida.
"""
)

b) COVARIANZA Y CORRELACIÓN ENTRE CO Y NOX
Covarianza Cov(CO, NOX)  : 11.5867
Correlación Corr(CO, NOX): 0.8656

¿Son distintas?
Sí, son métricas distintas:
1. La COVARIANZA mide la dirección de la relación lineal entre las variables en las unidades originales de medición (ppm × ppb). Su magnitud depende totalmente de las escalas de CO y NOX.
2. La CORRELACIÓN es una versión estandarizada/adimensional de la covarianza (restringida al intervalo [-1, 1]). Permite comparar la fuerza de la asociación lineal de manera independiente a las unidades de medida.



In [6]:
# c) Eigenvalores con y sin registros faltantes

# 1. Matriz de correlación eliminando observaciones con NaNs por pares (sin descartar todo el renglón)
corr_mat_nans = pivoted.corr()
eigvals_nans, _ = np.linalg.eig(corr_mat_nans)

# 2. Matriz de correlación eliminando completamente todas las filas que tengan al menos un faltante (complete cases)
pivoted_complete = pivoted.dropna()
corr_mat_complete = pivoted_complete.corr()
eigvals_complete, _ = np.linalg.eig(corr_mat_complete)

# Ordenar de mayor a menor para presentación
eigvals_nans = np.sort(eigvals_nans)[::-1]
eigvals_complete = np.sort(eigvals_complete)[::-1]

print("==================================================")
print("c) EIGENVALORES DE LA MATRIZ DE CORRELACIÓN")
print("==================================================")
print(
    "Eigenvalores (Calculados descartando filas con NaNs completos en el dataset):"
)
for i, val in enumerate(eigvals_complete, 1):
    print(f"  Eigenvalor {i}: {val:.6f}")

print("\nEigenvalores (Calculados con correlación por pares entre variables):")
for i, val in enumerate(eigvals_nans, 1):
    print(f"  Eigenvalor {i}: {val:.6f}")

print(
    "\nNota: La presencia de valores extremadamente cercanos a 0 confirma la redundancia lineal casi perfecta entre NO, NO2 y NOX."
)

c) EIGENVALORES DE LA MATRIZ DE CORRELACIÓN
Eigenvalores (Calculados descartando filas con NaNs completos en el dataset):
  Eigenvalor 1: 4.455667
  Eigenvalor 2: 1.978269
  Eigenvalor 3: 0.912216
  Eigenvalor 4: 0.644521
  Eigenvalor 5: 0.552580
  Eigenvalor 6: 0.334203
  Eigenvalor 7: 0.122036
  Eigenvalor 8: 0.000378
  Eigenvalor 9: 0.000131

Eigenvalores (Calculados con correlación por pares entre variables):
  Eigenvalor 1: 4.303344
  Eigenvalor 2: 1.992863
  Eigenvalor 3: 0.918607
  Eigenvalor 4: 0.657686
  Eigenvalor 5: 0.558406
  Eigenvalor 6: 0.428957
  Eigenvalor 7: 0.143673
  Eigenvalor 8: 0.000074
  Eigenvalor 9: -0.003612

Nota: La presencia de valores extremadamente cercanos a 0 confirma la redundancia lineal casi perfecta entre NO, NO2 y NOX.


In [7]:
# d) Modelo para predecir PM2.5

# Modelo 1: PM2.5 contra todos los demás contaminantes
df_reg_all = pivoted_complete.copy()
y_pm = df_reg_all["PM2.5"]
X_all = df_reg_all.drop(columns=["PM2.5"])
X_all_const = sm.add_constant(X_all)

model_all = sm.OLS(y_pm, X_all_const).fit()

vif_all = pd.DataFrame(
    {
        "Variable": X_all_const.columns,
        "VIF": [
            variance_inflation_factor(X_all_const.values, i)
            for i in range(X_all_const.shape[1])
        ],
    }
)

# Modelo 2: PM2.5 contra únicamente contaminantes gaseosos
gases = ["CO", "NO", "NO2", "NOX", "O3", "SO2"]
X_gases = df_reg_all[gases]
X_gases_const = sm.add_constant(X_gases)

model_gases = sm.OLS(y_pm, X_gases_const).fit()

vif_gases = pd.DataFrame(
    {
        "Variable": X_gases_const.columns,
        "VIF": [
            variance_inflation_factor(X_gases_const.values, i)
            for i in range(X_gases_const.shape[1])
        ],
    }
)

print("==================================================")
print("d) MODELOS DE REGRESIÓN PARA PM2.5")
print("==================================================")
print(f"1) PM2.5 vs Todos los Contaminantes -> R² = {model_all.rsquared:.4f}")
print("VIF - Modelo con Todos:")
print(vif_all.to_string(index=False))

print(
    f"\n2) PM2.5 vs Solo Gases ({', '.join(gases)}) -> R² = {model_gases.rsquared:.4f}"
)
print("VIF - Modelo Solo Gases:")
print(vif_gases.to_string(index=False))

print(
    "\nInterpretación: Al incluir PM10 y PMCO (que contienen físicamente a las partículas PM2.5), el R² asciende al 99.8%. Al usar únicamente gases, el R² cae a 40.5%."
)

d) MODELOS DE REGRESIÓN PARA PM2.5
1) PM2.5 vs Todos los Contaminantes -> R² = 0.9983
VIF - Modelo con Todos:
Variable         VIF
   const    1.000000
      CO    5.710228
      NO 2324.236785
     NO2  780.733295
     NOX 4528.146814
      O3    1.773441
    PM10    4.543673
    PMCO    3.403099
     SO2    1.109554

2) PM2.5 vs Solo Gases (CO, NO, NO2, NOX, O3, SO2) -> R² = 0.4055
VIF - Modelo Solo Gases:
Variable         VIF
   const    1.000000
      CO    5.583881
      NO 2324.147253
     NO2  780.533733
     NOX 4527.795725
      O3    1.337073
     SO2    1.043589

Interpretación: Al incluir PM10 y PMCO (que contienen físicamente a las partículas PM2.5), el R² asciende al 99.8%. Al usar únicamente gases, el R² cae a 40.5%.


In [8]:
# e) Matriz H y Leverage con datos de 1 estación y 1 semana
# Seleccionamos la estación 'PED' (Pedregal) durante una semana (168 horas)
estacion_e = "PED"
df_ped_week = (
    df[
        (df["id_station"] == estacion_e)
        & (df["id_parameter"].isin(["PM2.5"] + gases))
    ]
    .pivot(index="date", columns="id_parameter", values="valor")
    .dropna()
)

df_ped_week = df_ped_week.iloc[:168]  # 168 horas = 1 semana

X_week = sm.add_constant(df_ped_week[gases])
X_mat = X_week.values
p = X_mat.shape[1]  # Número de parámetros (incluyendo el intercepto)

# 1. Construcción explícita de la Matriz Hat: H = X(X^T X)^{-1} X^T
H = X_mat @ np.linalg.inv(X_mat.T @ X_mat) @ X_mat.T

# Comprobación 1: H^2 = H (Idempotencia)
H2 = H @ H
diff_idempotence = np.max(np.abs(H2 - H))

# Comprobación 2: traza(H) = p
trace_H = np.trace(H)

# 2. Leverage directo sin construir la matriz completa H: h_ii = diag(X(X^T X)^{-1} X^T)
# Usando la propiedad h_ii = sum_j (X_ij * (X (X^T X)^-1)_ij)
leverage_direct = np.sum(
    X_mat * (X_mat @ np.linalg.inv(X_mat.T @ X_mat)), axis=1
)
diff_leverage = np.max(np.abs(np.diag(H) - leverage_direct))

print("==================================================")
print(f"e) MATRIZ SOMBRERO (H) Y LEVERAGE - Estación {estacion_e} (1 Semana)")
print("==================================================")
print(f"Número de parámetros (p): {p}")
print(f"Diferencia máxima entre H² y H (Idempotencia): {diff_idempotence:.2e}")
print(f"Traza de H: {trace_H:.4f} (Coincide con p = {p})")
print(
    f"Diferencia máxima entre diag(H) y Leverage directo: {diff_leverage:.2e}"
)

e) MATRIZ SOMBRERO (H) Y LEVERAGE - Estación PED (1 Semana)
Número de parámetros (p): 7
Diferencia máxima entre H² y H (Idempotencia): 6.80e-14
Traza de H: 7.0000 (Coincide con p = 7)
Diferencia máxima entre diag(H) y Leverage directo: 1.73e-16


In [9]:
# f) Distancia de Cook y comparación con Leverage
y_week = df_ped_week["PM2.5"]
model_week = sm.OLS(y_week, X_week).fit()

# Obtener Distancia de Cook
influence = model_week.get_influence()
cooks_d = influence.cooks_distance[0]

# Identificar los 10 puntos con mayor Distancia de Cook y los 10 con mayor Leverage
top10_cook_idx = np.argsort(cooks_d)[-10:]
top10_lev_idx = np.argsort(leverage_direct)[-10:]

coincidentes = set(top10_cook_idx).intersection(set(top10_lev_idx))

print("==================================================")
print("f) DIAGNÓSTICO CON DISTANCIA DE COOK Y REAJUSTE")
print("==================================================")
print(f"Índices de los 10 puntos con mayor Cook:     {list(top10_cook_idx)}")
print(f"Índices de los 10 puntos con mayor Leverage: {list(top10_lev_idx)}")
print(f"Puntos coincidentes en ambos top 10:         {list(coincidentes)}")

print(
    """\n¿Los 10 más influyentes son los de más leverage?
No necesariamente todos. El Leverage mide únicamente qué tan extrema/alejada es la combinación de predictores (X), mientras que la Distancia de Cook evalúa la influencia real combinando el leverage con la magnitud del residuo (error de predicción en Y). Solo los puntos con alto leverage Y residuo grande son altamente influyentes."""
)

# Reajustar el modelo eliminando los 10 puntos más influyentes según Cook
df_ped_clean = df_ped_week.drop(df_ped_week.index[top10_cook_idx])
X_clean = sm.add_constant(df_ped_clean[gases])
y_clean = df_ped_clean["PM2.5"]

model_clean = sm.OLS(y_clean, X_clean).fit()

print("\n--- COMPARACIÓN DE REAJUSTE ---")
print(f"R² Modelo Original (Semana completa): {model_week.rsquared:.4f}")
print(f"R² Modelo Reajustado (Sin Top 10 Cook): {model_clean.rsquared:.4f}")

f) DIAGNÓSTICO CON DISTANCIA DE COOK Y REAJUSTE
Índices de los 10 puntos con mayor Cook:     [np.int64(95), np.int64(155), np.int64(12), np.int64(7), np.int64(2), np.int64(9), np.int64(96), np.int64(103), np.int64(10), np.int64(11)]
Índices de los 10 puntos con mayor Leverage: [np.int64(0), np.int64(96), np.int64(150), np.int64(126), np.int64(127), np.int64(125), np.int64(32), np.int64(124), np.int64(104), np.int64(103)]
Puntos coincidentes en ambos top 10:         [np.int64(96), np.int64(103)]

¿Los 10 más influyentes son los de más leverage?
No necesariamente todos. El Leverage mide únicamente qué tan extrema/alejada es la combinación de predictores (X), mientras que la Distancia de Cook evalúa la influencia real combinando el leverage con la magnitud del residuo (error de predicción en Y). Solo los puntos con alto leverage Y residuo grande son altamente influyentes.

--- COMPARACIÓN DE REAJUSTE ---
R² Modelo Original (Semana completa): 0.4496
R² Modelo Reajustado (Sin Top 10 Cook): 